In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler

# 设置绘图样式
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 加载单变量数据集
data_single = pd.read_csv('/mnt/e/平时作业/机器学习/实验1/ex1data1.txt', header=None, names=['Population', 'Profit'])
X_single = data_single.iloc[:, 0].values
y_single = data_single.iloc[:, 1].values
m_single = len(y_single)

# 添加截距项
X_single_b = np.c_[np.ones((m_single, 1)), X_single]

# 数据集可视化
plt.figure(figsize=(10, 6))
plt.scatter(X_single, y_single, c='red', marker='x', label='Training Data')
plt.xlabel('Population of City in 10,000s')
plt.ylabel('Profit in $10,000s')
plt.title('Scatter Plot of Training Data')
plt.legend()
plt.show()

In [ ]:
def compute_cost(X, y, theta):
    """
    计算线性回归的损失函数
    
    参数:
    X : 特征矩阵 (m x n+1)
    y : 目标向量 (m x 1)
    theta : 参数向量 (n+1 x 1)
    
    返回:
    cost : 损失函数值
    """
    m = len(y)
    predictions = X.dot(theta)
    errors = predictions - y
    cost = (1/(2*m)) * np.sum(errors**2)
    return cost

# 初始化参数并计算初始损失
theta_init = np.zeros(2)
initial_cost = compute_cost(X_single_b, y_single, theta_init)
print(f"初始损失值: {initial_cost:.4f}")

In [ ]:
def gradient_descent(X, y, theta, alpha, iterations):
    """
    执行梯度下降算法
    
    参数:
    X : 特征矩阵 (m x n+1)
    y : 目标向量 (m x 1)
    theta : 初始参数向量 (n+1 x 1)
    alpha : 学习率
    iterations : 迭代次数
    
    返回:
    theta : 优化后的参数向量
    cost_history : 每次迭代的损失值记录
    """
    m = len(y)
    cost_history = np.zeros(iterations)
    
    for i in range(iterations):
        predictions = X.dot(theta)
        errors = predictions - y
        gradient = (1/m) * X.T.dot(errors)
        theta = theta - alpha * gradient
        cost_history[i] = compute_cost(X, y, theta)
        
    return theta, cost_history
    # 开始迭代
    for i in range(1, iterations):
        # 计算预测值和梯度
        predictions = X.dot(theta)
        errors = predictions - y
        gradient = (1/m) * X.T.dot(errors)
        
        # 更新参数
        theta = theta - alpha * gradient
        
        # 计算新损失
        current_cost = compute_cost(X, y, theta)
        
        # 检查是否溢出
        if np.isinf(current_cost) or current_cost > max_cost:
            converged = False
            theta = theta + alpha * gradient  # 回退一步到有效参数
            cost_history[i:] = np.nan  # 标记后续迭代为无效
            break
            
        cost_history[i] = current_cost
        
    return theta, cost_history, converged

# 设置梯度下降参数
alpha_single = 0.01
iterations_single = 1500
theta_single, cost_history_single = gradient_descent(
    X_single_b, y_single, theta_init, alpha_single, iterations_single
)

print(f"优化后的参数: θ0 = {theta_single[0]:.4f}, θ1 = {theta_single[1]:.4f}")
print(f"最终损失值: {compute_cost(X_single_b, y_single, theta_single):.4f}")

In [ ]:
# 绘制回归线
plt.figure(figsize=(10, 6))
plt.scatter(X_single, y_single, c='red', marker='x', label='Training Data')
plt.plot(X_single, X_single_b.dot(theta_single), 'b-', label='Linear Regression')
plt.xlabel('Population of City in 10,000s')
plt.ylabel('Profit in $10,000s')
plt.title('Linear Regression Fit')
plt.legend()
plt.show()

# 预测示例
population1 = 3.5
profit1 = np.array([1, population1]).dot(theta_single)
print(f"人口为35,000的城市预测利润: ${profit1 * 10000:.2f}")

population2 = 7.0
profit2 = np.array([1, population2]).dot(theta_single)
print(f"人口为70,000的城市预测利润: ${profit2 * 10000:.2f}")

In [ ]:
# 创建参数网格
theta0_vals = np.linspace(-10, 10, 100)
theta1_vals = np.linspace(-1, 4, 100)
J_vals = np.zeros((len(theta0_vals), len(theta1_vals)))

# 计算网格上的损失值
for i in range(len(theta0_vals)):
    for j in range(len(theta1_vals)):
        t = np.array([theta0_vals[i], theta1_vals[j]])
        J_vals[i, j] = compute_cost(X_single_b, y_single, t)

# 绘制损失函数曲面
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
theta0_mesh, theta1_mesh = np.meshgrid(theta0_vals, theta1_vals)
surf = ax.plot_surface(theta0_mesh, theta1_mesh, J_vals.T, 
                       cmap=cm.coolwarm, alpha=0.6)
ax.set_xlabel(r'$\theta_0$')
ax.set_ylabel(r'$\theta_1$')
ax.set_zlabel('Cost J')
ax.view_init(30, 230)
plt.title('Cost Function Surface')
fig.colorbar(surf, shrink=0.5, aspect=5)
plt.show()

In [ ]:
# 绘制损失函数等高线
plt.figure(figsize=(10, 6))
contour = plt.contour(theta0_vals, theta1_vals, J_vals.T, 
                      levels=np.logspace(-2, 3, 20))
plt.clabel(contour, inline=True, fontsize=8)
plt.plot(theta_single[0], theta_single[1], 'rx', markersize=10, linewidth=2)
plt.xlabel(r'$\theta_0$')
plt.ylabel(r'$\theta_1$')
plt.title('Cost Function Contours')
plt.show()

In [ ]:
def analyze_iterations(X, y, alpha, iterations_list, max_cost=1e10):
    """
    分析不同迭代次数对最终损失的影响
    
    参数:
    X : 特征矩阵
    y : 目标向量
    alpha : 学习率
    iterations_list : 迭代次数列表
    max_cost : 最大允许损失值
    
    返回:
    results : 包含所有结果的字典
    """
    final_costs = []
    theta_init = np.zeros(X.shape[1])
    converged_flags = []
    
    plt.figure(figsize=(12, 8))
    
    # 为每个迭代次数运行梯度下降
    for max_iters in iterations_list:
        _, cost_history, converged = robust_gradient_descent(
            X, y, theta_init, alpha, max_iters, max_cost
        )
        
        # 获取最终有效损失值
        valid_costs = cost_history[~np.isnan(cost_history)]
        if len(valid_costs) > 0:
            final_cost = valid_costs[-1]
            # 记录收敛状态
            converged_flags.append(converged)
        else:
            final_cost = np.nan
            converged_flags.append(False)
            
        final_costs.append(final_cost)
    
    # 绘制结果
    plt.plot(iterations_list, final_costs, 'bo-', markersize=8, label=f'α={alpha}')
    
    # 标记未收敛点
    for i, converged in enumerate(converged_flags):
        if not converged:
            plt.scatter(iterations_list[i], final_costs[i], 
                       color='red', s=100, marker='x', zorder=10)
    
    plt.xlabel('Number of Iterations', fontsize=12)
    plt.ylabel('Final Cost Value', fontsize=12)
    plt.title('Effect of Iterations on Final Cost (with Convergence Check)', fontsize=14)
    plt.xscale('log')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    
    # 返回结果
    return {
        'iterations': iterations_list,
        'final_costs': final_costs,
        'converged': converged_flags
    }

# 使用安全的学习率（根据上一步分析选择合适值）
alpha_safe = 0.01  # 从之前分析中选取的最佳值

# 迭代次数列表（注意：排除可能导致溢出的过大值）
iterations_list = [5, 10, 20, 50, 100, 200, 500, 1000, 1500]

# 进行分析
results_iters = analyze_iterations(X_single_b, y_single, alpha_safe, iterations_list)

In [ ]:
# 增强的学习率影响分析（防止溢出错误）
def analyze_learning_rates(X, y, alphas, iterations=100, max_cost=1e10):
    """
    分析不同学习率对梯度下降收敛的影响
    
    参数:
    X : 特征矩阵
    y : 目标向量
    alphas : 学习率列表
    iterations : 最大迭代次数
    max_cost : 最大允许损失值
    
    返回:
    results : 包含所有结果的字典
    """
    results = {}
    plt.figure(figsize=(14, 8))
    
    # 为每个学习率运行梯度下降
    for alpha in alphas:
        theta_init = np.zeros(X.shape[1])
        _, cost_history, converged = robust_gradient_descent(
            X, y, theta_init, alpha, iterations, max_cost
        )
        
        # 可视化有效损失值
        valid_indices = ~np.isnan(cost_history)
        valid_cost = cost_history[valid_indices]
        
        if len(valid_cost) > 0:
            # 绘制有效迭代的损失曲线
            plt.plot(np.where(valid_indices)[0], valid_cost, 
                     label=f'α = {alpha}', linewidth=2)
            
            # 标记终止点（如果提前终止）
            if not converged:
                plt.scatter(len(valid_cost) - 1, valid_cost[-1], 
                           color='red', s=80, marker='x', zorder=10)
        
        results[alpha] = {
            'final_cost': valid_cost[-1] if len(valid_cost) > 0 else np.inf,
            'converged': converged,
            'iterations_needed': len(valid_cost)
        }
    
    plt.xlabel('Iterations', fontsize=12)
    plt.ylabel('Cost J', fontsize=12)
    plt.title('Robust Analysis: Effect of Learning Rate on Convergence', fontsize=14)
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    
    # 创建表格对比不同学习率的结果
    print("\n学习率对比分析:")
    print("α\t收敛情况\t最终损失\t所需迭代次数")
    for alpha in alphas:
        res = results[alpha]
        converged = "是" if res['converged'] else "否"
        cost = res['final_cost']
        iter_needed = res['iterations_needed']
        
        # 格式化输出
        cost_str = f"{cost:.4f}" if np.isfinite(cost) and res['converged'] else "发散"
        
        print(f"{alpha:.3f}\t{converged}\t\t{cost_str}\t{iter_needed}")
    
    return results

# 定义更合理的学习率范围
alphas_single = [0.001, 0.003, 0.01, 0.03, 0.05, 0.07, 0.1, 0.3]

# 进行鲁棒性分析
results_single = analyze_learning_rates(X_single_b, y_single, alphas_single, iterations=100)

In [ ]:
# 加载多变量数据集
data_multi = pd.read_csv('/mnt/e/平时作业/机器学习/实验1/ex1data2.txt', header=None, names=['Size', 'Bedrooms', 'Price'])
X_multi = data_multi[['Size', 'Bedrooms']].values
y_multi = data_multi['Price'].values
m_multi = len(y_multi)

# 特征归一化
scaler = StandardScaler()
X_multi_scaled = scaler.fit_transform(X_multi)

# 添加截距项
X_multi_b = np.c_[np.ones((m_multi, 1)), X_multi_scaled]

print("归一化后的特征前5行:")
print(X_multi_scaled[:5])
print("\n目标变量前5个值:")
print(y_multi[:5])

In [ ]:
# 创建3D散点图
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# 绘制数据点
scatter = ax.scatter(X_multi[:, 0], X_multi[:, 1], y_multi, 
                     c=y_multi, cmap='viridis', marker='o', alpha=0.8)

ax.set_xlabel('House Size (sq.ft)')
ax.set_ylabel('Number of Bedrooms')
ax.set_zlabel('Price ($)')
ax.set_title('House Prices by Size and Bedrooms')
fig.colorbar(scatter, label='Price ($)')
plt.show()

# 绘制特征与目标变量的关系
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 房屋面积与价格
ax1.scatter(X_multi[:, 0], y_multi, c='blue', alpha=0.6)
ax1.set_xlabel('House Size (sq.ft)')
ax1.set_ylabel('Price ($)')
ax1.set_title('Price vs. House Size')

# 卧室数量与价格
ax2.scatter(X_multi[:, 1], y_multi, c='red', alpha=0.6)
ax2.set_xlabel('Number of Bedrooms')
ax2.set_ylabel('Price ($)')
ax2.set_title('Price vs. Bedrooms')

plt.tight_layout()
plt.show()

In [ ]:
def compute_cost_multi(X, y, theta):
    """
    计算多变量线性回归的损失函数
    
    参数:
    X : 特征矩阵 (m x n+1)
    y : 目标向量 (m x 1)
    theta : 参数向量 (n+1 x 1)
    
    返回:
    cost : 损失函数值
    """
    m = len(y)
    predictions = X.dot(theta)
    cost = (1/(2*m)) * np.sum((predictions - y)**2)
    return cost

# 初始化参数并计算初始损失
theta_init_multi = np.zeros(X_multi_b.shape[1])
initial_cost_multi = compute_cost_multi(X_multi_b, y_multi, theta_init_multi)
print(f"初始损失值 (多变量): {initial_cost_multi:.2f}")

In [ ]:
def gradient_descent_multi(X, y, theta, alpha, iterations):
    """
    执行多变量梯度下降算法
    
    参数:
    X : 特征矩阵 (m x n+1)
    y : 目标向量 (m x 1)
    theta : 初始参数向量 (n+1 x 1)
    alpha : 学习率
    iterations : 迭代次数
    
    返回:
    theta : 优化后的参数向量
    cost_history : 每次迭代的损失值记录
    """
    m = len(y)
    cost_history = np.zeros(iterations)
    
    for i in range(iterations):
        predictions = X.dot(theta)
        errors = predictions - y
        gradient = (1/m) * X.T.dot(errors)
        theta = theta - alpha * gradient
        cost_history[i] = compute_cost_multi(X, y, theta)
        
    return theta, cost_history

# 设置梯度下降参数
alpha_multi = 0.01
iterations_multi = 1000
theta_multi, cost_history_multi = gradient_descent_multi(
    X_multi_b, y_multi, theta_init_multi, alpha_multi, iterations_multi
)

print(f"优化后的参数: {theta_multi}")
print(f"最终损失值: {cost_history_multi[-1]:.2f}")

In [ ]:
# 修改后的3D拟合图函数
def plot_3d_fitting(X1, X2, y, theta, scaler):
    """
    在3D散点图上绘制回归平面（修复警告版）
    
    参数:
    X1 - 房屋面积特征向量
    X2 - 卧室数量特征向量
    y - 实际房价向量
    theta - 优化后的参数向量
    scaler - 用于特征归一化的缩放器
    """
    # 创建3D图形
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # 设置视角
    ax.view_init(elev=25, azim=130)
    
    # 绘制原始数据点（增加大小和深度排序）
    scatter = ax.scatter(X1, X2, y, 
                         c=y, cmap='viridis', marker='o', 
                         s=50, alpha=0.9, depthshade=True,
                         label='Actual Prices')
    
    # 创建用于绘制平面的网格
    x1_min, x1_max = min(X1), max(X1)
    x2_min, x2_max = min(X2), max(X2)
    
    # 创建更精细的网格
    x1_range = np.linspace(x1_min, x1_max, 50)
    x2_range = np.linspace(x2_min, x2_max, 15)  # 卧室数量通常只有几个值
    
    # 将网格点组合成矩阵
    X1_grid, X2_grid = np.meshgrid(x1_range, x2_range)
    
    # 准备预测平面的数据
    Y_pred = np.zeros_like(X1_grid)
    
    # 对网格中的每个点进行预测（优化代码）
    # 展平网格点，一次性计算预测值
    flat_X1 = X1_grid.flatten()
    flat_X2 = X2_grid.flatten()
    
    # 创建特征矩阵
    samples = np.column_stack((flat_X1, flat_X2))
    
    # 归一化特征（使用训练时的缩放器）
    samples_scaled = scaler.transform(samples)
    
    # 添加截距项
    samples_b = np.c_[np.ones(samples_scaled.shape[0]), samples_scaled]
    
    # 一次性计算所有预测
    flat_preds = samples_b.dot(theta)
    
    # 重塑为网格形状
    Y_pred = flat_preds.reshape(X1_grid.shape)
    
    # 绘制拟合平面（使用双色调色板）
    plane = ax.plot_surface(X1_grid, X2_grid, Y_pred, 
                           cmap='coolwarm', alpha=0.6,
                           rstride=1, cstride=1,
                           label='Regression Plane')
    
    # 设置标签和标题
    ax.set_xlabel('House Size (sq.ft)', fontsize=12)
    ax.set_ylabel('Number of Bedrooms', fontsize=12)
    ax.set_zlabel('Price ($)', fontsize=12)
    ax.set_title('House Price Prediction with Fitted Regression Plane', fontsize=16)
    
    # 添加图例
    proxy = plt.Rectangle((0, 0), 1, 1, fc='crimson', alpha=0.6)
    ax.legend([proxy, scatter], ['Regression Plane', 'Actual Prices'], loc='best')
    
    # 添加颜色条
    cbar_scatter = fig.colorbar(scatter, ax=ax, pad=0.1)
    cbar_scatter.set_label('Actual Price ($)', rotation=270, labelpad=15)
    
    cbar_plane = fig.colorbar(plane, ax=ax, pad=0.15)
    cbar_plane.set_label('Predicted Price ($)', rotation=270, labelpad=15)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 提取原始特征用于可视化
X1 = X_multi[:, 0]  # 房屋面积
X2 = X_multi[:, 1]  # 卧室数量

# 创建3D拟合效果图
plot_3d_fitting(X1, X2, y_multi, theta_multi, scaler)

In [ ]:
# 尝试不同的学习率
alphas_multi = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
iterations = 100
plt.figure(figsize=(12, 8))

for alpha in alphas_multi:
    theta_temp = np.zeros(X_multi_b.shape[1])
    _, cost_history = gradient_descent_multi(
        X_multi_b, y_multi, theta_temp, alpha, iterations
    )
    plt.plot(range(iterations), cost_history, 
             label=f'α = {alpha}', linewidth=2)

plt.xlabel('Iterations')
plt.ylabel('Cost J')
plt.title('Effect of Learning Rate on Convergence (Multiple Variables)')
plt.yscale('log')
plt.legend()
plt.grid(True)
plt.show()

# 观察不同迭代次数下的损失变化
iterations_list = [10, 50, 100, 200, 500, 1000, 2000]
final_costs_multi = []

for iters in iterations_list:
    theta_temp = np.zeros(X_multi_b.shape[1])
    _, cost_history = gradient_descent_multi(
        X_multi_b, y_multi, theta_temp, alpha_multi, iters
    )
    final_costs_multi.append(cost_history[-1])

plt.figure(figsize=(10, 6))
plt.plot(iterations_list, final_costs_multi, 'bo-', markersize=8)
plt.xlabel('Number of Iterations')
plt.ylabel('Final Cost')
plt.title('Effect of Iterations on Final Cost (Multiple Variables)')
plt.xscale('log')
plt.grid(True)
plt.show()

In [ ]:
# 预测1650平方英尺、3个卧室的房屋价格
def predict_price(size, bedrooms, theta, scaler):
    """
    预测房屋价格
    
    参数:
    size : 房屋面积 (平方英尺)
    bedrooms : 卧室数量
    theta : 训练好的参数向量
    scaler : 用于特征归一化的缩放器
    
    返回:
    price : 预测价格
    """
    # 归一化特征
    features = np.array([size, bedrooms]).reshape(1, -1)
    features_scaled = scaler.transform(features)
    # 添加截距项
    features_b = np.c_[np.ones(1), features_scaled]
    # 预测价格
    return features_b.dot(theta)[0]

# 预测并显示结果
house_size = 1650
bedrooms = 3
predicted_price = predict_price(house_size, bedrooms, theta_multi, scaler)
print(f"预测1650平方英尺、3个卧室的房屋价格: ${predicted_price:.2f}")

# 与原始数据比较
print("\n原始数据中类似房屋的价格示例:")
similar_houses = data_multi[
    (data_multi['Size'] > 1600) & 
    (data_multi['Size'] < 1700) & 
    (data_multi['Bedrooms'] == 3)
]
print(similar_houses)